# 30. RNNs, sequence modeling, and encoder-decoder ideas

**Level:** Intermediate

**Prerequisites:** 27-28: neural layers and training.

[Course index](00_START_HERE.ipynb) | [Previous](29_Convolutional_Neural_Networks.ipynb) | [Next](31_Transfer_Learning_and_Frozen_Features.ipynb)

**Topics:** RNN, LSTM, GRU, Sequences, Sequence-to-sequence, Teacher forcing.

**Matching script:** `examples\30_RNNs_Sequences_and_Seq2Seq.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

A recurrent network updates a hidden state while reading ordered observations. Unlike an
unordered feature vector, a sequence has a meaningful time/token axis. RNNs reuse the same
transition parameters across positions; LSTM and GRU add gates that help control information flow.

A sequence-to-sequence model maps an input sequence to an output sequence, possibly of different
length. A classical encoder summarizes input and a decoder generates outputs conditioned on it.
Attention can let the decoder consult multiple encoder states instead of one bottleneck vector.
Modern transformer encoder-decoder systems use attention rather than recurrence for these roles.

Teacher forcing supplies true previous target tokens during decoder training. Inference instead
uses generated tokens, so training/inference conditions differ. Padding, sequence lengths,
masking, and causal direction must be explicit. Do not let a future observation enter features
for a prediction that was supposed to be made earlier.

## Mathematics or data contract

A simple recurrence is h_t=tanh(W_x*x_t+W_h*h_(t-1)+b).

## Important limitation

The shape exercise below is untrained; its output is not a meaningful translation or forecast.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Inspect recurrent output and hidden-state shapes

**Purpose:** Trace batch, sequence, and hidden dimensions.

### Read the code in this order
1. Create a batch-first three-sequence input.
2. Run a small GRU encoder.
3. Map every timestep to vocabulary logits.
4. Verify that final hidden state matches the last valid timestep for equal-length inputs.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
encoder: (3, 5, 6) hidden: (1, 3, 6)
per-position vocabulary logits: (3, 5, 8)
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(42)
inputs = torch.randn(3, 5, 4)
encoder = nn.GRU(input_size=4, hidden_size=6, batch_first=True)
outputs, hidden = encoder(inputs)
projection = nn.Linear(6, 8)
token_logits = projection(outputs)
assert outputs.shape == (3, 5, 6)
assert hidden.shape == (1, 3, 6) and token_logits.shape == (3, 5, 8)
torch.testing.assert_close(hidden[0], outputs[:, -1])
print("encoder:", tuple(outputs.shape), "hidden:", tuple(hidden.shape))
print("per-position vocabulary logits:", tuple(token_logits.shape))

encoder: (3, 5, 6) hidden: (1, 3, 6)
per-position vocabulary logits: (3, 5, 8)


### Interpret example 1

The result should demonstrate this purpose: **Trace batch, sequence, and hidden dimensions.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Add padded sequences of unequal lengths and use packed sequences or masks to preserve valid lengths.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `RNN` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: A simple recurrence is h_t=tanh(W_x*x_t+W_h*h_(t-1)+b).
- Name the boundary of the demonstration: The shape exercise below is untrained; its output is not a meaningful translation or forecast.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://docs.pytorch.org/tutorials/beginner/basics/intro.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.